# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?
Decision supported: a content strategist prioritizing limited review hours across a large portfolio.
Unit of analysis: one page, for one client.
Cost of a wrong call: false positive = wasted review time; false negative = a declining page goes unnoticed until recovery is more expensive.

In [4]:
question = "Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?"
decision_supported = "Content strategist prioritizing a limited weekly review queue (~50 pages)"
print(question)
print(decision_supported)

Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?
Content strategist prioritizing a limited weekly review queue (~50 pages)


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [5]:
grain_check = duckdb.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM fact_content
    WHERE report_date BETWEEN '2026-02-01' AND '2026-05-31'
    GROUP BY client_hash_id, content_hash_id, report_date HAVING c > 1 LIMIT 5
""").df()
print("Duplicate page-days in feature/label window:", len(grain_check))

print("\nRows:", len(data_model))
print("Feature columns:", ["impressions_window", "clicks_window", "april_impressions", "april_clicks",
      "february_clicks", "click_through_rate", "weighted_position", "momentum", "active_days"])
print("\nExcluded fields and why:")
excluded = {
    "provider_used/model_used": "internal authoring metadata, not a performance signal",
    "is_deleted": "product/lifecycle flag, used only to filter, never as a feature",
    "May clicks/impressions": "define the label — including as features would be direct leakage",
    "client_hash_id/content_hash_id": "grouping/joining keys only, never predictive features",
}
for k, v in excluded.items():
    print(f"- {k}: {v}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate page-days in feature/label window: 0

Rows: 183345
Feature columns: ['impressions_window', 'clicks_window', 'april_impressions', 'april_clicks', 'february_clicks', 'click_through_rate', 'weighted_position', 'momentum', 'active_days']

Excluded fields and why:
- provider_used/model_used: internal authoring metadata, not a performance signal
- is_deleted: product/lifecycle flag, used only to filter, never as a feature
- May clicks/impressions: define the label — including as features would be direct leakage
- client_hash_id/content_hash_id: grouping/joining keys only, never predictive features


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [6]:
model_cols = ["impressions_window", "clicks_window", "april_impressions", "april_clicks",
              "february_clicks", "click_through_rate", "weighted_position", "momentum",
              "active_days", "click_through_rate_missing", "weighted_position_missing", "momentum_missing"]

X = data_model[model_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

# --- Signal audit (baseline design) ---
volume_check = duckdb.sql("""
    SELECT CASE WHEN impressions_window < 100 THEN 'low' WHEN impressions_window < 1000 THEN 'medium' ELSE 'high' END AS bucket,
           COUNT(*) n, AVG(declined) pct_declined FROM data_model GROUP BY bucket ORDER BY bucket
""").df()
print("Volume signal:\n", volume_check)

age_check = duckdb.sql("""
    SELECT CASE WHEN content_age_days < 90 THEN '<90d' WHEN content_age_days < 180 THEN '90-180d' ELSE '180d+' END AS bucket,
           COUNT(*) n, AVG(declined) pct_declined FROM data_model GROUP BY bucket ORDER BY bucket
""").df()
print("\nStaleness signal:\n", age_check)

ctr_median = data_model["click_through_rate"].median()
ctr_check = duckdb.sql(f"""
    SELECT CASE WHEN click_through_rate < {ctr_median} THEN 'below_median' ELSE 'above_median' END AS bucket,
           COUNT(*) n, AVG(declined) pct_declined FROM data_model GROUP BY bucket
""").df()
print("\nCTR signal:\n", ctr_check)

Volume signal:
    bucket      n  pct_declined
0    high  70848      0.368098
1     low  51891      0.036827
2  medium  60606      0.132017

Staleness signal:
     bucket      n  pct_declined
0    180d+  89939      0.206729
1  90-180d  32710      0.196454
2     <90d  60696      0.180770

CTR signal:
          bucket      n  pct_declined
0  above_median  91673      0.386439
1  below_median  91672      0.006163


In [7]:
# --- Client-grouped split (the honest validation design) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
overlap = len(set(data_model.iloc[train_idx]["client_hash_id"]) & set(data_model.iloc[test_idx]["client_hash_id"]))
print("Client overlap in grouped split (must be 0):", overlap)

y_test = y.iloc[test_idx].reset_index(drop=True)

Client overlap in grouped split (must be 0): 0


In [8]:
# --- Leakage check: train-without the one label-adjacent feature ---
rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf_grouped.fit(X.iloc[train_idx], y.iloc[train_idx])
scores_grouped = rf_grouped.predict_proba(X.iloc[test_idx])[:, 1]
auc_grouped = roc_auc_score(y_test, scores_grouped)

cols_no_april = [c for c in model_cols if c not in ["april_clicks", "clicks_window"]]
rf_no_april = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf_no_april.fit(X[cols_no_april].iloc[train_idx], y.iloc[train_idx])
auc_no_april = roc_auc_score(y_test, rf_no_april.predict_proba(X[cols_no_april].iloc[test_idx])[:, 1])

print(f"AUC with april_clicks: {auc_grouped:.4f} | without: {auc_no_april:.4f} | collapse: {auc_grouped-auc_no_april:.4f}")

AUC with april_clicks: 0.9302 | without: 0.9041 | collapse: 0.0262


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [9]:
data_model["baseline_score"] = data_model["impressions_window"] / data_model["impressions_window"].max()
baseline_test = data_model.iloc[test_idx]["baseline_score"].reset_index(drop=True)

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
logreg_scores = logreg.predict_proba(X.iloc[test_idx])[:, 1]

p50_grouped = precision_at_k(y_test, scores_grouped, 50)

comparison = pd.DataFrame({
    "method": ["Base rate", "Baseline (volume)", "Logistic Regression", "Random Forest"],
    "precision@50": [
        y_test.mean(),
        precision_at_k(y_test, baseline_test.values, 50),
        precision_at_k(y_test, logreg_scores, 50),
        p50_grouped,
    ],
    "AUC": [
        0.5,
        roc_auc_score(y_test, baseline_test),
        roc_auc_score(y_test, logreg_scores),
        auc_grouped,
    ]
})
comparison

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,method,precision@50,AUC
0,Base rate,0.18879,0.500000
1,Baseline (volume),0.56000,0.785699
2,Logistic Regression,0.46000,0.867310
3,Random Forest,0.54000,0.930229


## 5. Limitations

*What this work cannot claim.*

In [10]:
zero_score_pct = (data_model["risk_score"] == 0.0).mean() if "risk_score" in data_model.columns else None
top5_client_share = data_model["client_hash_id"].value_counts().head(5).sum() / len(data_model)

limitations = [
    "No causal claim possible: no refresh was ever observed in this data.",
    f"Client concentration: top 5 clients = {top5_client_share:.1%} of the portfolio.",
    "Staleness signal inconsistent across time windows tested — not a stable universal rule.",
    "Cross-sectional, proxy-labeled data — all claims are observed/directional/decision-support only.",
]
for l in limitations:
    print("-", l)

- No causal claim possible: no refresh was ever observed in this data.
- Client concentration: top 5 clients = 56.6% of the portfolio.
- Staleness signal inconsistent across time windows tested — not a stable universal rule.
- Cross-sectional, proxy-labeled data — all claims are observed/directional/decision-support only.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [11]:
data_model["risk_score"] = rf_grouped.predict_proba(X)[:, 1]

def reason_code(row):
    if row["click_through_rate"] == 0 and row["impressions_window"] < 10:
        return "ZERO_TRAFFIC_DEAD_PAGE"
    high_volume = row["impressions_window"] >= data_model["impressions_window"].quantile(0.67)
    real_decline = (row["momentum_missing"] == 0) and (row["momentum"] <= -1.0)
    no_history = row["weighted_position_missing"] == 1
    if no_history: return "INSUFFICIENT_HISTORY"
    elif high_volume and real_decline: return "HIGH_VOLUME_AND_DECLINING"
    elif high_volume: return "HIGH_VOLUME_RISK"
    elif real_decline: return "SEVERE_MOMENTUM_DROP"
    return "MONITOR"

data_model["reason_code"] = data_model.apply(reason_code, axis=1)
data_model["action"] = data_model["reason_code"].map({
    "ZERO_TRAFFIC_DEAD_PAGE": "review_or_deprioritize", "INSUFFICIENT_HISTORY": "monitor_gather_data",
    "HIGH_VOLUME_AND_DECLINING": "refresh_now", "HIGH_VOLUME_RISK": "refresh",
    "SEVERE_MOMENTUM_DROP": "investigate", "MONITOR": "monitor"
})

print(data_model["reason_code"].value_counts(normalize=True))

ranked_queue = data_model.sort_values(["risk_score", "impressions_window"], ascending=[False, False])
diversified = ranked_queue.groupby("client_hash_id").head(5).sort_values("risk_score", ascending=False).head(50)
print("\nDistinct clients in final top-50 queue:", diversified["client_hash_id"].nunique())

reason_code
INSUFFICIENT_HISTORY         0.403442
HIGH_VOLUME_RISK             0.271428
MONITOR                      0.163326
ZERO_TRAFFIC_DEAD_PAGE       0.079986
SEVERE_MOMENTUM_DROP         0.042652
HIGH_VOLUME_AND_DECLINING    0.039167
Name: proportion, dtype: float64

Distinct clients in final top-50 queue: 17


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [12]:
os.makedirs("work/outputs", exist_ok=True)

diversified[["client_hash_id", "content_hash_id", "risk_score", "reason_code", "action"]].to_csv(
    "work/outputs/action_playbook_queue.csv", index=False
)

metrics_receipt = {
    "baseline": {"auc": float(roc_auc_score(y_test, baseline_test)), "precision_at_50": float(precision_at_k(y_test, baseline_test.values, 50))},
    "random_forest_may": {"auc": float(auc_grouped), "precision_at_50": float(p50_grouped)},
    "leakage_check_collapse": float(auc_grouped - auc_no_april),
    "top5_client_share": float(top5_client_share),
    "reason_code_distribution": data_model["reason_code"].value_counts(normalize=True).to_dict(),
}

with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(metrics_receipt, f, indent=2, default=str)

print(json.dumps(metrics_receipt, indent=2, default=str))

{
  "baseline": {
    "auc": 0.7856986695656439,
    "precision_at_50": 0.56
  },
  "random_forest_may": {
    "auc": 0.9302290957526194,
    "precision_at_50": 0.54
  },
  "leakage_check_collapse": 0.026168145865391845,
  "top5_client_share": 0.5660694319452398,
  "reason_code_distribution": {
    "INSUFFICIENT_HISTORY": 0.40344159917096184,
    "HIGH_VOLUME_RISK": 0.2714281818429736,
    "MONITOR": 0.16332597016553493,
    "ZERO_TRAFFIC_DEAD_PAGE": 0.0799858190842401,
    "SEVERE_MOMENTUM_DROP": 0.04265183124710246,
    "HIGH_VOLUME_AND_DECLINING": 0.03916659848918705
  }
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.